In [ ]:
import tensorflow as tf
import keras_tuner as kt
import os

from hyperparameter_search import CustomHyperModel
from load_data import load_datasets
from helper_functions import serialise_hpsearch
%matplotlib inline

# we want to use mixed precision so
tf.keras.mixed_precision.set_global_policy(tf.keras.mixed_precision.Policy('mixed_float16'))
# use newer memory allocator (added to avoid some oom errors)
# see: https://docs.nvidia.com/deeplearning/frameworks/tensorflow-user-guide/index.html
os.environ['TF_GPU_ALLOCATOR']='cuda_malloc_async'

In [ ]:
# first set some initial data up
dataset_path='datasets/130kv2/'
class_map = {0:"Original", 1:"Poisoned"}

# expected image resolution (all samples should have equal width/height), if set to a lower value samples will be resized accordingly
image_size=512
# batch size to use
batch_size=32

In [ ]:
# load our training, validation and test datasets
ds = load_datasets(dataset_path, class_map, batch_size, image_size, preview=True)

In [ ]:
# lets try a hyperparameter search on the dataset without resizing first
tuner = kt.BayesianOptimization(
    hypermodel=CustomHyperModel(input_shape=(image_size,image_size) + (3,), num_classes=2),
    objective='val_accuracy',
    max_trials=150,
    max_model_size=3e6, # added this to try and avoid us ending up with massively oversized models and OOM errors
    num_initial_points=None, # might want to play with this as our model has somewhat excessive dimensionality
    alpha=1e-3,
    beta=3.,
    seed=123,
    max_retries_per_trial=1, # we'll retry once but these are probably memory errors
    max_consecutive_failed_trials=3,
    directory='autotune',
    project_name='bayesopt_size512_batch32v3'
)
# tuner.search_space_summary()

# set up the callback for early stopping
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5)
]

# and then begin the search
tuner.search(
    ds['train'],
    epochs=10,
    callbacks=[callbacks],
    validation_data=ds['validation'],
)

# lastly we'll save the top 5 results to file
serialise_hpsearch(tuner,num_to_save=5)

In [ ]:
# now lets repeat this, but redo the datasets at 256x256 and a batch size of 64
image_size=256
batch_size=64

# reload our training, validation and test datasets
ds = load_datasets(dataset_path, class_map, batch_size, image_size)

tuner = kt.BayesianOptimization(
    hypermodel=CustomHyperModel(input_shape=(image_size,image_size) + (3,), num_classes=2),
    objective='val_accuracy',
    max_trials=150,
    num_initial_points=None, # might want to play with this as our model has somewhat excessive dimensionality
    alpha=1e-3,
    beta=3.,
    seed=123,
    max_retries_per_trial=1, # we'll retry once but these are probably memory errors
    max_consecutive_failed_trials=3,
    directory='autotune',
    project_name='bayesopt_crop256_batch32v3'
)

# tuner.search_space_summary()

# set up the callback for early stopping
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=5)
]

# and then begin the search
tuner.search(
    ds['train'],
    epochs=10,
    callbacks=[callbacks],
    validation_data=ds['validation'],
)

# lastly we'll save the top 5 results to file
serialise_hpsearch(tuner,num_to_save=5)

In [ ]:
"""
# see https://www.tensorflow.org/tutorials/keras/keras_tuner
tuner = kt.Hyperband(
    CustomHyperModel(input_shape=(image_size,image_size) + (3,), num_classes=2),
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='autotune',
    project_name='hyperband'
)
"""

In [ ]:
"""
epochs = 10

callbacks = [
    tf.keras.callbacks.ModelCheckpoint("models/save_at_{epoch}.keras"),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
]
model_history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
).history
"""

In [ ]:
"""
#model = tf.keras.models.load_model('save_at_3.keras')
eval_results = eval_model('initial_model',model,model_history,val_ds,test_ds)
#model_history
"""